# 89 — Audit Trail Assistant

## Goal
Turn **evidence logs** and **system events** into a
**structured audit narrative** — a compliance-ready report
that connects events, identifies gaps, and highlights anomalies.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Pydantic AI Agents** | Create agents with structured output types |
| **Tool binding** | `@agent.tool_plain` for data-lookup functions |
| **Structured output** | `output_type=BaseModel` for typed responses |
| **Audit timeline** | Chronological reconstruction from logs |

## Stack
- `pydantic-ai` — agent framework with structured output
- `pydantic` — data validation and BaseModel schemas
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
from datetime import datetime

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
warnings.filterwarnings("ignore")

from pydantic import BaseModel, Field
from pydantic_ai import Agent

print("All imports OK")

## Step 1 — Simulated Audit Logs

We'll simulate system access logs, change records, and approval events
for a **financial system access review** audit.

In [ ]:
# Simulated audit evidence: system access logs
ACCESS_LOGS = [
    {"timestamp": "2024-10-01 08:15:00", "user": "jsmith", "action": "LOGIN",
     "system": "FinanceERP", "ip": "10.0.1.45", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 08:22:00", "user": "jsmith", "action": "VIEW_REPORT",
     "system": "FinanceERP", "detail": "Q3 Revenue Report", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 09:00:00", "user": "admin_root", "action": "MODIFY_PERMISSIONS",
     "system": "FinanceERP", "detail": "Elevated jsmith to ADMIN role", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 09:05:00", "user": "jsmith", "action": "EXPORT_DATA",
     "system": "FinanceERP", "detail": "Exported all_transactions_2024.csv (45MB)", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 09:06:00", "user": "jsmith", "action": "EXPORT_DATA",
     "system": "FinanceERP", "detail": "Exported vendor_payments_2024.csv (12MB)", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 09:10:00", "user": "admin_root", "action": "MODIFY_PERMISSIONS",
     "system": "FinanceERP", "detail": "Reverted jsmith to USER role", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 14:00:00", "user": "mwilson", "action": "LOGIN",
     "system": "FinanceERP", "ip": "10.0.1.88", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 14:05:00", "user": "mwilson", "action": "APPROVE_PAYMENT",
     "system": "FinanceERP", "detail": "Approved PO-2024-889 ($145,000)", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 22:30:00", "user": "jsmith", "action": "LOGIN",
     "system": "FinanceERP", "ip": "203.45.67.89", "status": "SUCCESS"},
    {"timestamp": "2024-10-01 22:35:00", "user": "jsmith", "action": "MODIFY_VENDOR",
     "system": "FinanceERP", "detail": "Changed bank account for Vendor V-4451", "status": "SUCCESS"},
]

# Approval records
APPROVAL_RECORDS = [
    {"change_id": "CHG-001", "requester": "jsmith", "approver": "admin_root",
     "type": "PERMISSION_CHANGE", "approved": True, "approval_time": "2024-10-01 08:58:00",
     "justification": "End-of-quarter reporting needs"},
    {"change_id": "CHG-002", "requester": "mwilson", "approver": "cfoster",
     "type": "PAYMENT_APPROVAL", "approved": True, "approval_time": "2024-10-01 13:55:00",
     "justification": "Quarterly vendor payment per contract"},
]

# Employee directory
EMPLOYEE_DIR = {
    "jsmith": {"name": "John Smith", "department": "Finance", "role": "Senior Analyst",
               "normal_hours": "08:00-18:00", "location": "NYC Office"},
    "admin_root": {"name": "System Admin (Shared)", "department": "IT", "role": "SysAdmin",
                    "normal_hours": "24/7", "location": "N/A"},
    "mwilson": {"name": "Maria Wilson", "department": "Finance", "role": "VP Finance",
                "normal_hours": "07:00-19:00", "location": "NYC Office"},
}

print(f"Loaded {len(ACCESS_LOGS)} access log entries")
print(f"Loaded {len(APPROVAL_RECORDS)} approval records")
print(f"Loaded {len(EMPLOYEE_DIR)} employee records")

## Step 2 — Define Output Schema

In [ ]:
class AuditFinding(BaseModel):
    """A single audit finding."""
    finding_id: str = Field(description="Unique finding identifier e.g. F-001")
    severity: str = Field(description="HIGH, MEDIUM, or LOW")
    title: str = Field(description="Brief finding title")
    description: str = Field(description="Detailed finding description")
    evidence: str = Field(description="Supporting evidence from logs")
    recommendation: str = Field(description="Recommended remediation")

class AuditReport(BaseModel):
    """Structured audit report."""
    audit_title: str = Field(description="Title of the audit")
    audit_period: str = Field(description="Period covered")
    executive_summary: str = Field(description="2-3 sentence executive summary")
    findings: list[AuditFinding] = Field(description="List of audit findings")
    overall_risk_rating: str = Field(description="Overall risk: HIGH, MEDIUM, or LOW")

print("Schemas defined: AuditFinding, AuditReport")

## Step 3 — Build the Pydantic AI Audit Agent

In [ ]:
audit_agent = Agent(
    "ollama:qwen3.5:9b",
    output_type=AuditReport,
    instructions="""You are an internal audit assistant. Analyze system access logs,
approval records, and employee data to produce a structured audit report.

Focus on:
1. Unusual access patterns (off-hours, external IPs)
2. Privilege escalation events
3. Large data exports without proper justification
4. Segregation of duties violations
5. Shared or generic admin accounts
6. Vendor master data changes

Be thorough but factual. Cite specific log entries as evidence.
Do NOT wrap your response in thinking tags."""
)

# Tool: look up access logs for a user
@audit_agent.tool_plain
def get_user_access_logs(username: str) -> str:
    """Retrieve access logs for a specific user"""
    logs = [e for e in ACCESS_LOGS if e["user"] == username]
    if not logs:
        return f"No access logs found for {username}"
    return json.dumps(logs, indent=2)

# Tool: look up employee info
@audit_agent.tool_plain
def get_employee_info(username: str) -> str:
    """Retrieve employee directory information"""
    info = EMPLOYEE_DIR.get(username)
    if not info:
        return f"No employee record found for {username}"
    return json.dumps(info, indent=2)

# Tool: look up approval records
@audit_agent.tool_plain
def get_approval_records(requester: str) -> str:
    """Retrieve change approval records for a user"""
    records = [r for r in APPROVAL_RECORDS if r["requester"] == requester]
    if not records:
        return f"No approval records found for {requester}"
    return json.dumps(records, indent=2)

# Tool: get all unique users from logs
@audit_agent.tool_plain
def list_all_users() -> str:
    """List all unique users found in access logs"""
    users = list(set(e["user"] for e in ACCESS_LOGS))
    return json.dumps(users)

print("Audit agent created with 4 tools")

## Step 4 — Run the Audit

In [ ]:
# Prepare the audit evidence summary for the agent
evidence_summary = f"""Conduct an access review audit for the FinanceERP system.

AUDIT PERIOD: October 1, 2024

Use the available tools to examine user access patterns, employee info,
and approval records. Look for anomalies, policy violations, and risks.

Key areas to examine:
- All users who accessed the system
- Any privilege changes and their approvals
- Data export activities
- Off-hours access
- Vendor master data changes

ALL ACCESS LOG ENTRIES FOR REFERENCE:
{json.dumps(ACCESS_LOGS, indent=2)}"""

print("Running audit agent...")
print("(This may take a minute as the agent calls tools and reasons)\n")

result = audit_agent.run_sync(evidence_summary)
report = result.output

print("Audit complete!")
print(f"Type of result: {type(report).__name__}")

In [ ]:
# Display the structured report
print("=" * 60)
print(f"AUDIT REPORT: {report.audit_title}")
print(f"Period: {report.audit_period}")
print(f"Overall Risk: {report.overall_risk_rating}")
print("=" * 60)
print(f"\nEXECUTIVE SUMMARY:\n{report.executive_summary}")

print(f"\n{'=' * 60}")
print(f"FINDINGS ({len(report.findings)} total)")
print("=" * 60)

for finding in report.findings:
    severity_icon = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}.get(finding.severity, "⚪")
    print(f"\n{severity_icon} [{finding.finding_id}] {finding.title}")
    print(f"   Severity: {finding.severity}")
    print(f"   Description: {finding.description}")
    print(f"   Evidence: {finding.evidence}")
    print(f"   Recommendation: {finding.recommendation}")

In [ ]:
# Export report as JSON (machine-readable audit artifact)
report_json = report.model_dump()
print("\nStructured JSON output:")
print(json.dumps(report_json, indent=2))

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Pydantic AI Agent** | `Agent('ollama:qwen3.5:9b', output_type=AuditReport)` |
| **Structured output** | `BaseModel` schema guarantees typed fields |
| **Tool binding** | `@agent.tool_plain` for log / employee lookups |
| **Synchronous invocation** | `agent.run_sync(prompt)` |
| **Audit narrative** | Findings, evidence, and recommendations |

## Pydantic AI vs LangChain/LangGraph

| Feature | Pydantic AI | LangChain/LangGraph |
|---------|------------|--------------------|
| **Output typing** | Native via `output_type` | Manual JSON parsing |
| **Tool definition** | Simple decorator `@agent.tool_plain` | Requires `@tool` + schema |
| **Validation** | Built-in Pydantic validation | Custom validation code |
| **Complexity** | Lightweight, focused | Full framework, more features |

## Extending This Project
- Connect to real SIEM (Splunk, Elastic) via API tools
- Add SOX compliance rule checking
- Multi-period trend analysis
- PDF report generation with findings and charts